In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error,root_mean_squared_error,confusion_matrix
from catboost import CatBoostRegressor
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split 


# Data loading

In [2]:

results_calculation = {}
ignore_filters = [
  "Source",
  "Destination",
  'ts_gps_source',
  'ts_gps_destination']
# Transform categorical features
categorical_features = [
  "Received Packets",
  "Packet_error_ratio"
]
TARGET_COLUMN = "pdr" 

In [3]:
def preprocess_nan_values(dataFrame,filters = []):
    """Replace NaN in categorical columns with empty string and maintain proper dtypes"""
   # Convert back to categorical type
    dataFrame = dataFrame.copy()
        

    for col in dataFrame.columns:
        if col in filters:
            if pd.api.types.is_string_dtype(dataFrame[col]) or pd.api.types.is_object_dtype(dataFrame[col]):
                dataFrame[col] = dataFrame[col].fillna('Unknown').astype(str)
        else:
            dataFrame[col] = dataFrame[col].astype(float)
            if col == 'SNR':
                for row in dataFrame[col]:
                    if row < 0:
                        dataFrame[col] = dataFrame[col].replace(row, 1)
    
    return dataFrame



def preprocess_category_values(dataFrame,filters = []):
   # Convert back to categorical type
    dataFrame = dataFrame.copy()
        

    for col in dataFrame.columns:
        if col in filters:
            if pd.api.types.is_float_dtype(dataFrame[col]):
                dataFrame[col] = dataFrame[col].round().astype('int64').astype('category')
            else:
                dataFrame[col] = dataFrame[col].fillna('Unknown').astype('category')
        else:
            if pd.api.types.is_datetime64_any_dtype(dataFrame[col]):
                dataFrame[col] = dataFrame[col].astype('int64') / 1e9
            else:
                dataFrame[col] = dataFrame[col].astype(float).abs()
    
    return dataFrame

In [4]:
cellular_df = preprocess_nan_values(pd.read_csv('cellular_df.csv'),ignore_filters)
cellular_df.drop(columns=ignore_filters,inplace=True)


## Feature Segregation and Importances

In [5]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso
def plot_feature_distribution(dataFrame, features,target):
    plt.figure(figsize=(15, 10))
    # Plot relationships between key network metrics
    # Accept both float and category dtypes for plotting
    valid_features = []
    for f in features:
        if isinstance(dataFrame[f], pd.Float64Dtype) or isinstance(dataFrame[f], pd.Categorical):
            valid_features.append(f)
        else:
            valid_features = features
    sns.pairplot(dataFrame[valid_features])
    plt.suptitle('Feature Relationships', y=1.02)
    plt.show()

    # 2. Feature Importance Analysis
    # Preprocessing
    df = dataFrame.copy()

    # Handle missing values
    imputer = SimpleImputer(strategy='median')
    df_imputed = pd.DataFrame(imputer.fit_transform(df.select_dtypes(include=['float64'])), 
                            columns=df.select_dtypes(include=['float64']).columns)

    X = df_imputed.drop(columns=[target, 'id'])  # Exclude identifiers
    y = df_imputed[target]

    # Train Random Forest model
    model = RandomForestRegressor(n_estimators=150, random_state=42)
    model.fit(X, y)

    # Get feature importances
    feature_importances = pd.DataFrame({
        'Feature': X.columns,
        'Importance': model.feature_importances_
    }).sort_values(by='Importance', ascending=False)

    # Plot feature importance
    plt.figure(figsize=(12, 8))
    sns.barplot(x='Importance', y='Feature', data=feature_importances)
    plt.title('Feature Importance for '+target.title()+' Prediction')
    plt.xlabel('Importance Score')
    plt.ylabel('Features')
    plt.show()

    # Display top features
    print("Top 10 Important Features:")
    print(feature_importances.head(10))
    
show_feature_distribution = False
if show_feature_distribution:
    plot_feature_distribution(cellular_df.head(10000), categorical_features,TARGET_COLUMN)

# Data preparation

In [6]:
# Set up category values to categorical features
cellular_df = preprocess_category_values(cellular_df,categorical_features)

# Split inputs and targets
X = cellular_df.drop(columns=[TARGET_COLUMN])
y = cellular_df[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)

X_train_copy = X_train.copy()
X_test_copy = X_test.copy()
y_train_copy = y_train.copy()
y_test_copy = y_test.copy()

# Scaling data using MinMax Scalar
I have a data of around 10000+ so I need to scale the models so that they fall under specific range to lessen the errors


In [7]:
from sklearn.preprocessing import MinMaxScaler
x_scaler = MinMaxScaler()
y_scaler = MinMaxScaler()



# Prediction algorithm using Decision Tree Regressor

In [8]:
dt_regressor = DecisionTreeRegressor(
    max_depth=5,          # Control tree depth to prevent overfitting
    min_samples_split=10, # Minimum samples required to split a node
    random_state=42       # For reproducibility
)

dt_regressor.fit(X_train, y_train)  # Train on the training data
y_test_dt =dt_regressor.predict(X_test).reshape(-1, 1)


# Compute error metric for Decision Tree Regressor

In [9]:

r2 = r2_score(y_test, y_test_dt)
# Calculating RMSE,MAE for DTR predictions and normalizing it by the range of the target variable in reference [1]
rmse = list(root_mean_squared_error(y_test, y_test_dt,multioutput='raw_values'))
mae = mean_absolute_error(y_test, y_test_dt)
print(f"R²: {r2:.4f}")
print(f"RMSE: {rmse[0]:.4f}")
print(f"MAE: {mae:.4f}")

results_calculation['Decision Tree Regressor'] = {'r2':r2, 'rmse': rmse[0], 'mae':mae}



R²: 0.9951
RMSE: 0.0224
MAE: 0.0144


## Prediction using SVM

In [10]:

model = SVR(kernel='rbf', C=1,max_iter=10000,verbose=True,gamma=0.08)

X_train_scale = x_scaler.fit_transform(X_train)
X_test_scale = x_scaler.transform(X_test)
y_train_scale = y_scaler.fit_transform(y_train.values.reshape(-1, 1)).flatten()
y_test_scale = y_scaler.transform(y_test.values.reshape(-1, 1)).flatten()

model.fit(X_train_scale, y_train_scale)

[LibSVM]*.
*
optimization finished, #iter = 1675
obj = -8.767486, rho = -0.547917
nSV = 205, nBSV = 53


SVR(C=1, gamma=0.08, max_iter=10000, verbose=True)

In [11]:
y_pred_svr =model.predict(X_test_scale).reshape(-1, 1)

r2_svr = r2_score(y_test_scale, y_pred_svr)
rmse_svr = root_mean_squared_error(y_test_scale, y_pred_svr,multioutput='raw_values')
mae_svr = mean_absolute_error(y_test_scale, y_pred_svr)
print(f"SVR R²: {r2_svr:.4f}")
print(f"SVR RMSE: {rmse_svr[0]:.4f}")
print(f"SVR MAE: {mae_svr:.4f}")
results_calculation['Support Vector Machine'] = {'r2':r2_svr, 'rmse': rmse_svr[0], 'mae':mae_svr}

SVR R²: 0.9608
SVR RMSE: 0.0586
SVR MAE: 0.0524


# Prediction using XGBoost

In [12]:

model = XGBRegressor(
        n_estimators=5000,
        learning_rate=0.1,
        max_depth=5,
        random_state=42,
        enable_categorical=True
    )

model.fit(X_train, y_train)

y_pred_xg =model.predict(X_test).reshape(-1, 1)

# y_pred_xg_shifted = y_pred_xg - y_test.min() + 1


In [13]:

r2_xg = r2_score(y_test, y_pred_xg) 
# Calculating RMSE,MAE for XGBoost predictions and normalizing it by the range of the target variable in reference [1]
rmse_xg =root_mean_squared_error(y_test, y_pred_xg,multioutput='raw_values')
mae_xg = mean_absolute_error(y_test, y_pred_xg)
print(f"R²: {r2_xg:.4f}")
print(f"RMSE: {rmse_xg[0]:.3f}")
print(f"MAE: {mae_xg:.4f}")

results_calculation['XGBoost'] = {'r2':r2_xg, 'rmse': rmse_xg[0], 'mae':mae_xg}


R²: 1.0000
RMSE: 0.000
MAE: 0.0000


# Cat Boost Algorithm Prediction

In [14]:


cat_features_indices = [X_train_copy.columns.get_loc(col) for col in categorical_features]
from catboost import Pool

#POOLING
train_pool = Pool(
    data=X_train_copy,
    label=y_train_copy,
    cat_features=categorical_features
)

test_pool = Pool(
    data=X_test_copy,
    label=y_test_copy,
    cat_features=categorical_features
)

# Model for CatBoostAlgorithm 
model_cat = CatBoostRegressor(
    iterations=4000,
    learning_rate=0.05,
    depth=4,
    verbose=1000,
    cat_features=categorical_features,
    random_seed=42,
    early_stopping_rounds=10,
    l2_leaf_reg=10,
    loss_function='MAE',
    allow_writing_files=False
)



model_cat.fit(train_pool,eval_set=test_pool)

pred_cat = model_cat.predict(test_pool)






0:	learn: 0.2277660	test: 0.2295708	best: 0.2295708 (0)	total: 99.5ms	remaining: 6m 37s
1000:	learn: 0.0340455	test: 0.0344291	best: 0.0344291 (1000)	total: 15s	remaining: 44.9s
2000:	learn: 0.0240979	test: 0.0244707	best: 0.0244707 (2000)	total: 29.4s	remaining: 29.4s
3000:	learn: 0.0188039	test: 0.0191359	best: 0.0191359 (3000)	total: 44.3s	remaining: 14.8s
3999:	learn: 0.0156253	test: 0.0159304	best: 0.0159304 (3999)	total: 59.1s	remaining: 0us

bestTest = 0.0159303739
bestIteration = 3999



In [15]:
r2_cat = r2_score(y_test_copy, pred_cat)
# Calculating RMSE for CatBoost predictions and normalizing it by the range of the target variable in reference [1]
rmse_cat = root_mean_squared_error(y_test_copy, pred_cat, multioutput='raw_values')
mae_cat = mean_absolute_error(y_test_copy, pred_cat)

print(f"R²: {r2_cat:.4f}")
print(f"RMSE: {rmse_cat[0]:.4f}")
print(f"MAE: {mae_cat:.4f}")

results_calculation['CatBoost'] = {'r2':r2_cat, 'rmse':  rmse_cat[0], 'mae':mae_cat}


R²: 0.9854
RMSE: 0.0386
MAE: 0.0159


## CREATE A PLOTTING GRAPH .

In [16]:
results_calculation

{'Decision Tree Regressor': {'r2': 0.9950909375360634,
  'rmse': 0.022412794430119374,
  'mae': 0.014431148823338416},
 'Support Vector Machine': {'r2': 0.9608302938183204,
  'rmse': 0.05862031261460555,
  'mae': 0.052355652597023586},
 'XGBoost': {'r2': 0.999999326540739,
  'rmse': 0.0002625139540353173,
  'mae': 9.07586118026734e-06},
 'CatBoost': {'r2': 0.985438226779418,
  'rmse': 0.03860147149102979,
  'mae': 0.015931310146337533}}

In [17]:
import plotly.graph_objects as go

model_names = list(results_calculation.keys())
r2_values = [results_calculation[model]['r2'] for model in model_names]
rmse_values = [results_calculation[model]['rmse'] for model in model_names]
mae_values = [results_calculation[model]['mae'] for model in model_names]
print("RMSE values:", rmse_values)
print("R2 values:", r2_values)
print("MAE values:", mae_values)

# Creating the bar chart
fig = go.Figure()

# Adding R-Square (R²) bars
fig.add_trace(go.Bar(
    x=model_names,
    y=r2_values,
    name='R² Error',
    marker_color='blue'
))

# Adding Root Mean Squared Error (RMSE) bars
fig.add_trace(go.Bar(
    x=model_names,
    y=rmse_values,
    name='RMSE',
    marker_color='green'
))

# Adding MAE bars
fig.add_trace(go.Bar(
    x=model_names,
    y=mae_values,
    name='MAE',
    marker_color='red'
))

# Updating layout
fig.update_layout(
    title='Comparison of Model Performance Metrics',
    xaxis_title='Models',
    yaxis_title='Metric Values',
    barmode='group',
    legend_title='Metrics',
    yaxis=dict(
        tickformat=".5f", 
        range=[0, 0.1],
        dtick=0.05
    )
    
)

# Show the plot
fig.show()

RMSE values: [0.022412794430119374, 0.05862031261460555, 0.0002625139540353173, 0.03860147149102979]
R2 values: [0.9950909375360634, 0.9608302938183204, 0.999999326540739, 0.985438226779418]
MAE values: [0.014431148823338416, 0.052355652597023586, 9.07586118026734e-06, 0.015931310146337533]


References:

[1] Willmott, C.J., & Matsuura, K. (2005). Advantages of the mean absolute error (MAE) over the root mean square error (RMSE) in assessing average model performance. Climate Research, 30, 79-82.